# SASRec Stage 3 Attention Bias Multi-Task BPI2012 Colab Train 04

This notebook tests the Stage 3 attention-bias multi-task setting on the `anchor_ml20`
backbone with a smaller `time_loss_weight=0.1`.

Main comparison groups:
- `anchor_single_task`
- `anchor_attention_bias_single_task`
- `anchor_multi_task`
- `anchor_attention_bias_multi_task_w1.0`
- `anchor_attention_bias_multi_task_w0.1`

Main comparison metric:
- `full ranking + NDCG@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
MULTITASK_ATTNBIAS_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2'
MULTITASK_ATTNBIAS_W01_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_w01_ndcg10_v2'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR:', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('MULTITASK_ATTNBIAS_OUTPUT_DIR:', MULTITASK_ATTNBIAS_OUTPUT_DIR)
print('MULTITASK_ATTNBIAS_W01_OUTPUT_DIR:', MULTITASK_ATTNBIAS_W01_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2
MULTITASK_ATTNBIAS_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2
MULTITASK_ATTNBIAS_W01_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_w01_ndcg10_v2
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"
!mkdir -p "$MULTITASK_ATTNBIAS_OUTPUT_DIR"
!mkdir -p "$MULTITASK_ATTNBIAS_W01_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


## Prepare Stage 3 processed dataset

This notebook regenerates the Stage 3 dataset into the versioned Drive folder
before training, so the run does not depend on any stale local processed files.


In [8]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!ls "$DATA_DIR"


/content/time-aware-behavior-prediction
[info] moved existing output to backup: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2__backup_20260601_143104
[ok] regenerated Stage 3 processed dataset at: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
[ok] metadata written to: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2/stage3_dataset_metadata.json
{
  "timestamp": 0,
  "delta_prev_seconds": 0,
  "delta_start_seconds": 0,
  "delta_next_seconds": 0
}
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


## Verify Stage 3 processed file

Stage 3 next-time prediction uses the processed time-feature CSV.
This check confirms that `delta_next_seconds` already exists.


In [10]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = [
    'delta_prev_seconds',
    'delta_start_seconds',
    'delta_next_seconds',
]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

Stage 3 attention-bias multi-task follow-up runs:

- backbone: `anchor_ml20`
- time-aware backbone: `delta_start + 9-bucket attention bias`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- time loss weight: `0.1`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison uses all 3 seeds: `42`, `2024`, `7`
- direct comparison target: existing `time_loss_weight=1.0` runs


## Check prerequisite reference runs


In [11]:
from pathlib import Path

baseline_required_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]
multitask_baseline_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
]
multitask_attnbias_w10_runs = [
    'multitask_attnbias_dstart_ml20_b9_s42',
    'multitask_attnbias_dstart_ml20_b9_s2024',
    'multitask_attnbias_dstart_ml20_b9_s7',
]

checks = [
    ('baseline', BASELINE_NDCG10_OUTPUT_DIR, baseline_required_runs),
    ('single-task attention bias', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR, single_task_attnbias_runs),
    ('multi-task baseline', MULTITASK_BASELINE_OUTPUT_DIR, multitask_baseline_runs),
    ('multi-task attention bias w=1.0', MULTITASK_ATTNBIAS_OUTPUT_DIR, multitask_attnbias_w10_runs),
]

print('=' * 80)
for label, output_dir, run_names in checks:
    print(label)
    base = Path(output_dir)
    for run_name in run_names:
        run_dir = base / run_name
        print(' ', run_name, 'EXISTS' if run_dir.exists() else 'MISSING')
    print('-' * 80)


baseline
  anchor_ml20_s42 EXISTS
  anchor_ml20_s2024 EXISTS
  anchor_ml20_s7 EXISTS
--------------------------------------------------------------------------------
single-task attention bias
  attnbias_dstart_ml20_b9_s42 EXISTS
  attnbias_dstart_ml20_b9_s2024 EXISTS
  attnbias_dstart_ml20_b9_s7 EXISTS
--------------------------------------------------------------------------------
multi-task baseline
  multitask_anchor_ml20_s42 EXISTS
  multitask_anchor_ml20_s2024 EXISTS
  multitask_anchor_ml20_s7 EXISTS
--------------------------------------------------------------------------------
multi-task attention bias w=1.0
  multitask_attnbias_dstart_ml20_b9_s42 EXISTS
  multitask_attnbias_dstart_ml20_b9_s2024 EXISTS
  multitask_attnbias_dstart_ml20_b9_s7 EXISTS
--------------------------------------------------------------------------------


## Check planned attention-bias multi-task runs (`time_loss_weight=0.1`)


In [12]:
planned_attnbias_multitask_runs = [
    'multitask_attnbias_dstart_ml20_b9_w01_s42',
    'multitask_attnbias_dstart_ml20_b9_w01_s2024',
    'multitask_attnbias_dstart_ml20_b9_w01_s7',
]

output_dir = Path(MULTITASK_ATTNBIAS_W01_OUTPUT_DIR)
print('=' * 80)
print('Stage 3 attention-bias multi-task runs (time_loss_weight=0.1)')
for run_name in planned_attnbias_multitask_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 attention-bias multi-task runs (time_loss_weight=0.1)
multitask_attnbias_dstart_ml20_b9_w01_s42 OK
multitask_attnbias_dstart_ml20_b9_w01_s2024 OK
multitask_attnbias_dstart_ml20_b9_w01_s7 OK


## Train attention-bias multi-task runs (`time_loss_weight=0.1`)


In [13]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml20_b9_w01_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_ATTNBIAS_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_w01_ndcg10_v2/multitask_attnbias_dstart_ml20_b9_w01_s42
epoch=1, loss=1.0195
epoch=2, loss=0.4633
epoch=3, loss=0.3554
epoch=4, loss=0.3065
epoch=5, loss=0.2776
valid [task], Top5Acc: 0.5309, Top10Acc: 0.7653, Acc: 0.0863, MacroF1: 0.1414, TimeMAE: 72986.5739, TimeRMSE: 278918.7325, TimeMedAE: 5354.2000
valid [full], NDCG@5: 0.7331, HR@5: 0.9449, NDCG@10: 0.7511, HR@10: 0.9984, MRR: 0.6731
valid [sampled], NDCG@5: 0.5552, HR@5: 0.5555, NDCG@10: 0.5609, HR@10: 0.5741, MRR: 0.5756
test [task], Top5Acc: 0.4194, Top10Acc: 0.6693, Acc: 0.0314, MacroF1: 0

In [14]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml20_b9_w01_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_ATTNBIAS_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_w01_ndcg10_v2/multitask_attnbias_dstart_ml20_b9_w01_s2024
epoch=1, loss=0.9840
epoch=2, loss=0.4895
epoch=3, loss=0.3669
epoch=4, loss=0.3123
epoch=5, loss=0.2803
valid [task], Top5Acc: 0.3981, Top10Acc: 0.7330, Acc: 0.0732, MacroF1: 0.1501, TimeMAE: 76199.9513, TimeRMSE: 295566.1979, TimeMedAE: 2855.7133
valid [full], NDCG@5: 0.6434, HR@5: 0.7535, NDCG@10: 0.7163, HR@10: 0.9869, MRR: 0.6383
valid [sampled], NDCG@5: 0.5576, HR@5: 0.5577, NDCG@10: 0.5596, HR@10: 0.5639, MRR: 0.5719
test [task], Top5Acc: 0.2893, Top10Acc: 0.5802, Acc: 0.0266, MacroF1:

In [15]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml20_b9_w01_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_ATTNBIAS_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_w01_ndcg10_v2/multitask_attnbias_dstart_ml20_b9_w01_s7
epoch=1, loss=1.0562
epoch=2, loss=0.4929
epoch=3, loss=0.3764
epoch=4, loss=0.3206
epoch=5, loss=0.2869
valid [task], Top5Acc: 0.4837, Top10Acc: 0.7012, Acc: 0.1367, MacroF1: 0.1538, TimeMAE: 74213.9603, TimeRMSE: 289822.3759, TimeMedAE: 932.0979
valid [full], NDCG@5: 0.6437, HR@5: 0.7635, NDCG@10: 0.7056, HR@10: 0.9558, MRR: 0.6345
valid [sampled], NDCG@5: 0.5455, HR@5: 0.5456, NDCG@10: 0.5487, HR@10: 0.5556, MRR: 0.5612
test [task], Top5Acc: 0.3442, Top10Acc: 0.5342, Acc: 0.0331, MacroF1: 0.0

In [16]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_delta_column': config.get('time_delta_column'),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [17]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [18]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]
multitask_baseline_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
]
multitask_attnbias_w10_runs = [
    'multitask_attnbias_dstart_ml20_b9_s42',
    'multitask_attnbias_dstart_ml20_b9_s2024',
    'multitask_attnbias_dstart_ml20_b9_s7',
]
multitask_attnbias_w01_runs = [
    'multitask_attnbias_dstart_ml20_b9_w01_s42',
    'multitask_attnbias_dstart_ml20_b9_w01_s2024',
    'multitask_attnbias_dstart_ml20_b9_w01_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
single_task_attnbias_df = rebuild_df(SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
multitask_baseline_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)
multitask_attnbias_w10_df = rebuild_df(MULTITASK_ATTNBIAS_OUTPUT_DIR)
multitask_attnbias_w01_df = rebuild_df(MULTITASK_ATTNBIAS_W01_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = 'anchor_single_task'

single_task_attnbias_subset = single_task_attnbias_df[single_task_attnbias_df['run_name'].isin(single_task_attnbias_runs)].copy()
single_task_attnbias_subset['variant'] = 'anchor_attnbias_single_task'

multitask_baseline_subset = multitask_baseline_df[multitask_baseline_df['run_name'].isin(multitask_baseline_runs)].copy()
multitask_baseline_subset['variant'] = 'anchor_multi_task'

multitask_attnbias_w10_subset = multitask_attnbias_w10_df[multitask_attnbias_w10_df['run_name'].isin(multitask_attnbias_w10_runs)].copy()
multitask_attnbias_w10_subset['variant'] = 'anchor_attnbias_multi_task_w1.0'

multitask_attnbias_w01_subset = multitask_attnbias_w01_df[multitask_attnbias_w01_df['run_name'].isin(multitask_attnbias_w01_runs)].copy()
multitask_attnbias_w01_subset['variant'] = 'anchor_attnbias_multi_task_w0.1'

df_compare = pd.concat(
    [
        baseline_subset,
        single_task_attnbias_subset,
        multitask_baseline_subset,
        multitask_attnbias_w10_subset,
        multitask_attnbias_w01_subset,
    ],
    ignore_index=True,
)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

id_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric', 'best_epoch',
    'time_loss_weight', 'enable_time_prediction', 'time_modeling_mode',
]

metric_prefixes = (
    'best_valid_',
    'best_test_at_best_valid_',
    'last_valid_',
    'last_test_',
)
metric_cols = sorted([c for c in df_compare.columns if c.startswith(metric_prefixes)])
display_cols = [c for c in id_cols if c in df_compare.columns] + metric_cols

df_compare[display_cols]


,run_name,seed,variant,maxlen,dropout_rate,selection_metric,best_epoch,time_loss_weight,enable_time_prediction,time_modeling_mode,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_mean_rank,best_test_at_best_valid_full_median_rank,best_test_at_best_valid_full_mrr,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_num_eval_users,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_mean_rank,best_test_at_best_valid_sampled_median_rank,best_test_at_best_valid_sampled_mrr,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_num_eval_users,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_time_mae,best_test_at_best_valid_task_time_median_ae,best_test_at_best_valid_task_time_rmse,best_test_at_best_valid_task_top10_accuracy,best_test_at_best_valid_task_top1_accuracy,best_test_at_best_valid_task_top5_accuracy,best_valid_full_hr@10,best_valid_full_hr@5,best_valid_full_mean_rank,best_valid_full_median_rank,best_valid_full_mrr,best_valid_full_ndcg@10,best_valid_full_ndcg@5,best_valid_full_num_eval_users,best_valid_sampled_hr@10,best_valid_sampled_hr@5,best_valid_sampled_mean_rank,best_valid_sampled_median_rank,best_valid_sampled_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_ndcg@5,best_valid_sampled_num_eval_users,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_time_mae,best_valid_task_time_median_ae,best_valid_task_time_rmse,best_valid_task_top10_accuracy,best_valid_task_top1_accuracy,best_valid_task_top5_accuracy,last_test_full_hr@10,last_test_full_hr@5,last_test_full_mean_rank,last_test_full_median_rank,last_test_full_mrr,last_test_full_ndcg@10,last_test_full_ndcg@5,last_test_full_num_eval_users,last_test_sampled_hr@10,last_test_sampled_hr@5,last_test_sampled_mean_rank,last_test_sampled_median_rank,last_test_sampled_mrr,last_test_sampled_ndcg@10,last_test_sampled_ndcg@5,last_test_sampled_num_eval_users,last_test_task_accuracy,last_test_task_macro_f1,last_test_task_time_mae,last_test_task_time_median_ae,last_test_task_time_rmse,last_test_task_top10_accuracy,last_test_task_top1_accuracy,last_test_task_top5_accuracy,last_valid_full_hr@10,last_valid_full_hr@5,last_valid_full_mean_rank,last_valid_full_median_rank,last_valid_full_mrr,last_valid_full_ndcg@10,last_valid_full_ndcg@5,last_valid_full_num_eval_users,last_valid_sampled_hr@10,last_valid_sampled_hr@5,last_valid_sampled_mean_rank,last_valid_sampled_median_rank,last_valid_sampled_mrr,last_valid_sampled_ndcg@10,last_valid_sampled_ndcg@5,last_valid_sampled_num_eval_users,last_valid_task_accuracy,last_valid_task_macro_f1,last_valid_task_time_mae,last_valid_task_time_median_ae,last_valid_task_time_rmse,last_valid_task_top10_accuracy,last_valid_task_top1_accuracy,last_valid_task_top5_accuracy
0,multitask_attnbias_dstart_ml20_b9_w01_s7,7,anchor_attnbias_multi_task_w0.1,20,0.2,full_valid_ndcg@10,5,0.1,True,attention_bias,0.952407,0.822066,3.238102,2.0,0.532007,0.631334,0.588142,7396,0.247972,0.155084,24.813007,17.0,0.194492,0.180086,0.151288,7396,0.033126,0.024758,13085.986746,6.330000,85188.488546,0.534208,0.033126,0.344240,0.955782,0.763537,3.450476,1.0,0.634460,0.705648,0.643668,7350,0.555646,0.545578,21.437687,1.0,0.561173,0.548667,0.545501,7350,0.136735,0.153762,74213.960287,932.097898,289822.375947,0.701224,0.136735,0.483673,1.000000,0.998506,2.010865,1.0,0.714270,0.784455,0.783940,7363,0.401738,0.357599,13.908054,15.0,0.387031,0.366614,0.352753,7363,0.052288,0.032042,12675.970917,7.570000,71578.282649,0.669021,0.052288,0.260356,0.913592,0.804531,3.447097,2.0,0.611253,0.678269,0.643578,7372,0.517770,0.480060,21.732637,8.0,0.495531,0.487765,0.475830,7372,0.067146,0.084509,73759.328876,2708.506195,286960.044770,0.611774,0.067146,0.400977
1,multitask_attnbias_dstart_ml20_b9_w01_s42,42,anchor_attnbias_multi

In [19]:
summary_metric_cols = sorted([
    c for c in df_compare.columns
    if c.startswith((
        'best_valid_',
        'best_test_at_best_valid_',
        'last_valid_',
        'last_test_',
    ))
])

summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare


best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_mean_rank           best_test_at_best_valid_full_median_rank          best_test_at_best_valid_full_mrr           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_num_eval_users            best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_mean_rank           best_test_at_best_valid_sampled_median_rank           best_test_at_best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_num_eval_users            best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_median_ae            best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_top1_accuracy           best_test_at_best_valid_task_top5_accuracy           best_valid_full_hr@10           best_valid_full_hr@5           best_valid_full_mean_rank           best_valid_full_median_rank      best_valid_full_mrr           best_valid_full_ndcg@10           best_valid_full_ndcg@5           best_valid_full_num_eval_users            best_valid_sampled_hr@10           best_valid_sampled_hr@5           best_valid_sampled_mean_rank           best_valid_sampled_median_rank      best_valid_sampled_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_ndcg@5           best_valid_sampled_num_eval_users            best_valid_task_accuracy           best_valid_task_macro_f1           best_valid_task_time_mae              best_valid_task_time_median_ae              best_valid_task_time_rmse                \
                                                              mean       std                              mean       std                                   mean       std                                     mean      std                             mean       std                                 mean       std                                mean       std                                        mean        std                                  mean       std                                 mean       std                                      mean       std                                        mean       std                                mean       std                                    mean       std                                   mean       std                                           mean        std                                  mean       std                                  mean       std                                  mean          std                                        mean        std                                   mean          std                                        mean       std                                       mean       std                                       mean       std                  mean       std                 mean       std                      mean       std                        mean  std                mean       std                    mean       std                   mean       std                           mean        std                     mean       std                    mean       std                         mean       std                           mean  std                   mean       std                       mean       std                      mean       std                              mean        std                     mean       std                     mean       std                     mean          std                           mean          std                      mean           std   
variant                                             

Interpretation guide:

- compare `anchor_attnbias_multi_task_w1.0` vs `anchor_attnbias_multi_task_w0.1` first
- use `best_test_at_best_valid_full_ndcg@10` as the main decision metric
- check whether smaller `time_loss_weight` recovers activity performance while keeping time error reasonable
